In [2]:
import os
import cv2
import numpy as np
import pandas as pd
import easyocr
from deepface import DeepFace
from tqdm import tqdm
import warnings
import sys


26-06-06 14:12:00 - Directory C:\Users\Nathan\.deepface has been created
26-06-06 14:12:00 - Directory C:\Users\Nathan\.deepface\weights has been created


In [3]:
warnings.filterwarnings("ignore")

THUMBNAILS_DIR = '../thumbnails'    
OUTPUT_CSV = '../data/features_visuais.csv' 

print("Carregando modelo de OCR...")
reader = easyocr.Reader(['pt', 'en'], gpu=False)

Using CPU. Note: This module is much faster with a GPU.


Carregando modelo de OCR...
Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

In [4]:
def calculate_colorfulness(image):
    (B, G, R) = cv2.split(image.astype("float"))
    rg = np.absolute(R - G)
    yb = np.absolute(0.5 * (R + G) - B)
    rg_mean, rg_std = np.mean(rg), np.std(rg)
    yb_mean, yb_std = np.mean(yb), np.std(yb)
    std_root = np.sqrt((rg_std ** 2) + (yb_std ** 2))
    mean_root = np.sqrt((rg_mean ** 2) + (yb_mean ** 2))
    return std_root + (0.3 * mean_root)

In [5]:
def analyze_image(filename, filepath):
    video_id = filename.split('.')[0]
    
    image = cv2.imread(filepath)
    if image is None:
        return None
    
    height, width, _ = image.shape
    total_area = height * width

    # Métricas Estéticas
    colorfulness = calculate_colorfulness(image)
    hsv_image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    brightness = np.mean(hsv_image[:, :, 2])
    
    # Métricas de Texto (EasyOCR)
    text_results = reader.readtext(image)
    text_area = 0
    extracted_words = []
    
    for (bbox, text, prob) in text_results:
        if prob > 0.3:
            w = np.linalg.norm(np.array(bbox[0]) - np.array(bbox[1]))
            h = np.linalg.norm(np.array(bbox[0]) - np.array(bbox[3]))
            text_area += (w * h)
            extracted_words.append(text)
            
    text_ratio = text_area / total_area if total_area > 0 else 0
    full_text = " ".join(extracted_words)

    # Métricas Faciais (DeepFace)
    face_ratio = 0
    dominant_emotion = "None"
    has_extreme_emotion = 0
    
    try:
        faces = DeepFace.analyze(img_path=filepath, actions=['emotion'], enforce_detection=False, silent=True)
        face = faces[0] if isinstance(faces, list) else faces
            
        region = face.get('region', {})
        if region and region.get('w', 0) > 0:
            face_area = region['w'] * region['h']
            face_ratio = face_area / total_area
            
            dominant_emotion = face['dominant_emotion']
            if dominant_emotion in ['surprise', 'fear']:
                has_extreme_emotion = 1
                
    except Exception:
        pass

    return {
        'videoId': video_id,
        'colorfulness': round(colorfulness, 2),
        'brightness': round(brightness, 2),
        'face_ratio': round(face_ratio, 4),
        'dominant_emotion': dominant_emotion,
        'has_extreme_emotion': has_extreme_emotion,
        'text_ratio': round(text_ratio, 4),
        'thumbnail_text': full_text
    }

In [7]:
if not os.path.exists(THUMBNAILS_DIR):
    print(f"Erro: O diretório '{THUMBNAILS_DIR}' não existe.")
    sys.exit(1)

image_files = [f for f in os.listdir(THUMBNAILS_DIR) if f.endswith(('.jpg', '.png'))]

processed_ids = set()
write_header = True

if os.path.exists(OUTPUT_CSV):
    try:
        df_existing = pd.read_csv(OUTPUT_CSV, usecols=['videoId'])
        processed_ids = set(df_existing['videoId'].astype(str))
        write_header = False 
        print(f"Arquivo de saída detectado. {len(processed_ids)} thumbnails já processadas.")
    except Exception as e:
        print(f"Erro ao ler o CSV existente: {e}. Começando um novo.")

images_to_process = [f for f in image_files if f.split('.')[0] not in processed_ids]
total_images = len(images_to_process)

if total_images == 0:
    print("Todas as imagens já foram processadas!")
    sys.exit(1)
    
print(f"Iniciando extração para as {total_images} imagens restantes...")

BATCH_SIZE = 50  
lote_atual = []

for i, filename in enumerate(tqdm(images_to_process, desc="Processando Thumbnails")):
    filepath = os.path.join(THUMBNAILS_DIR, filename)
    
    feature_dict = analyze_image(filename, filepath)
    
    if feature_dict:
        lote_atual.append(feature_dict)
        
    if len(lote_atual) >= BATCH_SIZE or i == total_images - 1:
        if lote_atual: 
            df_batch = pd.DataFrame(lote_atual)
            df_batch.to_csv(OUTPUT_CSV, mode='a', index=False, header=write_header)
            
            write_header = False 
            lote_atual = []   

print("\n" + "="*40)
print(f"Extração concluída! Resultados atualizados em '{OUTPUT_CSV}'.")
print("="*40)

Arquivo de saída detectado. 271 thumbnails já processadas.
Iniciando extração para as 8608 imagens restantes...


Processando Thumbnails: 100%|██████████| 8608/8608 [6:32:43<00:00,  2.74s/it]   


Extração concluída! Resultados atualizados em '../data/features_visuais.csv'.
